In [ ]:
# Configuration
# Available GCMs (run once per model):
#   MPI-ESM1-2-HR, ACCESS-CM2, EC-Earth3, CNRM-ESM2-1,
#   BCC-CSM2-MR, MRI-ESM2-0, NorESM2-MM
#
# This notebook downloads PRMS_RAPID streamflow from HydroSource.
# VIC5_RAPID streamflow uses the same structure (change prefix accordingly).

BATCH_SCENARIOS = [126, 245, 370, 585]
SCENARIO = 585  # Used if BATCH_SCENARIOS is empty

# Time periods to process (both will be downloaded)
TIME_PERIODS = [(1980, 2019), (2020, 2059), (2060, 2099)]  # Or single: [(2060, 2099)]

# Watershed locations
LOCATIONS = [
    # Brazos Headwaters (1205)
    12050001, 12050002, 12050003, 12050004, 12050005, 12050006, 12050007,
    # Middle Brazos-Clear Fork (120601)
    12060101, 12060102, 12060103, 12060104, 12060105,
    # Middle Brazos-Bosque (120602)
    12060201, 12060202, 12060203, 12060204,
    # Lower Brazos (120701)
    12070101, 12070102, 12070103, 12070104,
    # Little River tributaries (120702)
    12070201, 12070202, 12070203, 12070204, 12070205
]

# Export format
EXPORT_FORMAT = 'parquet'  # Options: 'parquet', 'zarr', 'hdf5', 'csv'

# Download settings
MAX_WORKERS = 6
MAX_RETRIES = 3
TIMEOUT = 120
SKIP_EXISTING = True

# Data source
BASE_URL = "https://hydrosource2.ornl.gov/files/SWA9505V3Flow"
MODEL = "MRI-ESM2-0"
EXPERIMENT = "DBCCA_Daymet"

In [ ]:
import os
import time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import requests
import xarray as xr
import pandas as pd
from tqdm.auto import tqdm

In [ ]:
def download_file(location, base_url, save_dir, scenario, start_year, end_year, 
                  max_retries=3, timeout=120, skip_existing=True):
    filename = f"PRMS_RAPID_{location}N_{MODEL}_ssp{scenario}_r1i1p1f1_{EXPERIMENT}_{start_year}_{end_year}.nc"
    url = f"{base_url}/PRMS_RAPID_{MODEL}_ssp{scenario}_r1i1p1f1_{EXPERIMENT}_{start_year}_{end_year}/{filename}"
    filepath = os.path.join(save_dir, filename)
    
    if skip_existing and os.path.exists(filepath) and os.path.getsize(filepath) > 0:
        return filename, True, "Skipped"
    
    for attempt in range(max_retries):
        try:
            response = requests.get(url, stream=True, timeout=timeout)
            if response.status_code == 200:
                with open(filepath, 'wb') as f:
                    for chunk in response.iter_content(chunk_size=1024*1024):
                        if chunk:
                            f.write(chunk)
                return filename, True, "Downloaded"
            else:
                return filename, False, f"HTTP {response.status_code}"
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
            else:
                return filename, False, str(e)
    return filename, False, "Unknown error"


def download_all_files(locations, base_url, save_dir, scenario, start_year, end_year, max_workers=4, **kwargs):
    os.makedirs(save_dir, exist_ok=True)
    print(f"\nDownloading SSP{scenario} ({start_year}-{end_year}): {len(locations)} files")
    
    successful = failed = 0
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {
            executor.submit(download_file, loc, base_url, save_dir, scenario, start_year, end_year, **kwargs): loc 
            for loc in locations
        }
        with tqdm(total=len(locations), desc="Downloading") as pbar:
            for future in as_completed(futures):
                filename, success, message = future.result()
                if success:
                    successful += 1
                else:
                    failed += 1
                    pbar.write(f"✗ {filename}: {message}")
                pbar.update(1)
    
    print(f"Summary: {successful} successful, {failed} failed\n")
    return successful, failed


def process_nc_file(nc_file):
    try:
        ds = xr.open_dataset(nc_file)
        ds = ds.assign_coords({"time_dy": ds["Time_dy"], "comid": ds["COMID"]})
        df = ds["RAPID_dy_cfs"].transpose("time_dy", "comid").to_pandas()
        ds.close()
        return df
    except Exception as e:
        print(f"Error processing {nc_file}: {e}")
        return None


def export_dataframe(df, basename, format='parquet'):
    format = format.lower()
    start = time.time()
    
    if format == 'parquet':
        filepath = f"{basename}.parquet"
        df.to_parquet(filepath, compression='snappy', index=True)
    elif format == 'zarr':
        filepath = f"{basename}.zarr"
        ds = xr.Dataset({'streamflow': (['date', 'comid'], df.values)}, 
                        coords={'date': df.index, 'comid': df.columns})
        ds.chunk({'date': 365, 'comid': -1}).to_zarr(filepath, mode='w', consolidated=True)
    elif format == 'hdf5':
        filepath = f"{basename}.h5"
        df.to_hdf(filepath, key='streamflow', mode='w', complevel=9)
    elif format == 'csv':
        filepath = f"{basename}.csv"
        df.to_csv(filepath, index=True)
    else:
        raise ValueError(f"Unknown format: {format}")
    
    elapsed = time.time() - start
    size = os.path.getsize(filepath) if format != 'zarr' else sum(
        f.stat().st_size for f in Path(filepath).rglob('*') if f.is_file())
    print(f"Exported: {filepath} ({size/1e6:.1f} MB in {elapsed:.1f}s)")
    return filepath, size, elapsed


def combine_and_export(folder_path, output_basename, format='parquet'):
    import glob
    nc_files = glob.glob(os.path.join(folder_path, "*.nc"))
    if not nc_files:
        raise ValueError(f"No NC files found in {folder_path}")
    
    print(f"Processing {len(nc_files)} NC files...")
    dfs = []
    for nc_file in tqdm(nc_files, desc="Processing"):
        df = process_nc_file(nc_file)
        if df is not None:
            dfs.append(df)
    
    if not dfs:
        raise ValueError("No data successfully processed")
    
    df_combined = pd.concat(dfs, axis=1)
    df_combined.index = pd.to_datetime(df_combined.index.astype(str), format='%Y%m%d')
    df_combined.index.name = 'date'
    
    print(f"Combined: {df_combined.shape[0]:,} days × {df_combined.shape[1]:,} locations")
    print(f"Date range: {df_combined.index[0]} to {df_combined.index[-1]}")
    
    export_dataframe(df_combined, output_basename, format=format)
    return df_combined

In [ ]:
# Main execution
scenarios = BATCH_SCENARIOS if BATCH_SCENARIOS else [SCENARIO]
results = {}

print(f"Processing {len(scenarios)} scenarios × {len(TIME_PERIODS)} time periods = {len(scenarios)*len(TIME_PERIODS)} total")
print(f"Export format: {EXPORT_FORMAT.upper()}\n")

for scenario in scenarios:
    for start_year, end_year in TIME_PERIODS:
        print(f"\n{'='*70}")
        print(f"SSP{scenario}: {start_year}-{end_year}")
        print(f"{'='*70}")
        
        download_dir = f"PRMS_RAPID_{MODEL}_ssp{scenario}_r1i1p1f1_{EXPERIMENT}_{start_year}_{end_year}"
        output_basename = f"PRMS_RAPID_{MODEL}_ssp{scenario}_r1i1p1f1_{EXPERIMENT}_{start_year}_{end_year}_daily"
        
        # Download
        successful, failed = download_all_files(
            locations=LOCATIONS,
            base_url=BASE_URL,
            save_dir=download_dir,
            scenario=scenario,
            start_year=start_year,
            end_year=end_year,
            max_workers=MAX_WORKERS,
            max_retries=MAX_RETRIES,
            timeout=TIMEOUT,
            skip_existing=SKIP_EXISTING
        )
        
        # Process and export
        if successful > 0:
            df = combine_and_export(download_dir, output_basename, format=EXPORT_FORMAT)
            results[f'ssp{scenario}_{start_year}_{end_year}'] = df
            print(f"✓ Completed SSP{scenario} ({start_year}-{end_year})\n")
        else:
            print(f"✗ Failed SSP{scenario} ({start_year}-{end_year})\n")

print(f"\n{'='*70}")
print(f"COMPLETE: {len(results)} datasets processed")
print(f"{'='*70}")
for key in results.keys():
    print(f"  - results['{key}']")